# **Practical-3 Implement a feedforward neural network using TensorFlow or PyTorch.**

In [ ]:
## 1. Importing Libraries
import torch
import torch.nn as nn
import torch.optim as optim
# torch: Core PyTorch library for tensor computations and automatic differentiation.
# torch.nn: Contains building blocks for neural networks (layers, activation functions, loss functions).
# torch.optim: Provides optimization algorithms like Adam and SGD.

### Importing Libraries

* **`import torch`**: PyTorch is an open-source deep learning framework that provides tensor computations with GPU acceleration and automatic differentiation via its `autograd` engine — the backbone of training neural networks.
* **`import torch.nn as nn`**: The `nn` module contains all the building blocks needed to define a neural network: layers (`Linear`), activation functions (`ReLU`, `Sigmoid`), and loss functions (`BCELoss`).
* **`import torch.optim as optim`**: The `optim` module provides optimization algorithms that update the network's parameters during training. Here we use `Adam`, an efficient adaptive learning-rate optimizer.

In [ ]:
# 2. Defining the Dataset
X = torch.tensor([[0,0], [0,1], [1,0], [1,1]], dtype=torch.float32)
y = torch.tensor([[0], [0], [0], [1]], dtype=torch.float32)
# X is the input dataset representing all four combinations of AND gate inputs.
# y is the output label, where the output is 1 only when both inputs are 1, as per the AND gate truth table.

### Defining the Dataset

We are simulating the **AND logic gate** using a feedforward neural network.

| Input 1 | Input 2 | Output (AND) |
|---------|---------|----------------|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

* **`X`**: A `4x2` PyTorch tensor holding all four possible combinations of two binary inputs.
* **`y`**: A `4x1` PyTorch tensor holding the expected AND gate output for each input pair.
* **`dtype=torch.float32`**: Ensures the data is stored as 32-bit floating point numbers, which is required for PyTorch computations and compatible with the loss function.

In [ ]:
# 3. Define a Feedforward Neural Network
class FeedforwardNN(nn.Module):
    def __init__(self):
        super(FeedforwardNN, self).__init__()
        self.fc1 = nn.Linear(2, 4)   # Hidden layer: 2 inputs -> 4 neurons
        self.relu = nn.ReLU()         # ReLU activation function
        self.fc2 = nn.Linear(4, 1)   # Output layer: 4 neurons -> 1 output
        self.sigmoid = nn.Sigmoid()   # Sigmoid activation function

    def forward(self, x):
        out = self.relu(self.fc1(x))       # Pass through hidden layer + ReLU
        out = self.sigmoid(self.fc2(out))  # Pass through output layer + Sigmoid
        return out

### Defining the Feedforward Neural Network

* **`class FeedforwardNN(nn.Module)`**: We define our network as a Python class that inherits from `nn.Module` — PyTorch's base class for all neural network models. This gives the class automatic parameter tracking and gradient computation.
* **`__init__` method**: Defines the layers of the network as attributes:
  * **`nn.Linear(2, 4)`**: A fully connected (dense) hidden layer that takes 2 input features and outputs 4 values — one per hidden neuron.
  * **`nn.ReLU()`**: Rectified Linear Unit activation. Outputs the input directly if positive, otherwise 0. Introduces non-linearity so the network can learn beyond simple linear mappings.
  * **`nn.Linear(4, 1)`**: The output layer that takes the 4 hidden neuron outputs and reduces them to a single value.
  * **`nn.Sigmoid()`**: Squashes the output into the range `[0, 1]`, which can be interpreted as a probability — ideal for binary classification.
* **`forward` method**: Defines how data flows through the network. PyTorch calls this automatically when you pass input to the model.

In [ ]:
# 4. Initialize Model, Loss Function, and Optimizer
model = FeedforwardNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
# model: Creates an instance of the FeedforwardNN with randomly initialized weights.
# criterion: Binary Cross-Entropy Loss — standard loss for binary classification.
# optimizer: Adam optimizer with learning rate 0.01 to update model parameters.

### Initializing Model, Loss, and Optimizer

* **`model = FeedforwardNN()`**: Creates an instance of the network. PyTorch automatically initializes the weights and biases of each `Linear` layer randomly.
* **`criterion = nn.BCELoss()`**: Binary Cross-Entropy Loss is the standard loss function for binary classification problems. It measures how far the predicted probability is from the actual label (0 or 1) and guides the weight updates during backpropagation.
* **`optimizer = optim.Adam(model.parameters(), lr=0.01)`**: Adam (Adaptive Moment Estimation) is an efficient optimizer that adapts the learning rate for each parameter individually during training, generally leading to faster and more stable convergence than plain SGD.

In [ ]:
# 5. Training the Model
for epoch in range(100):
    outputs = model(X)           # Forward pass: compute predictions
    loss = criterion(outputs, y) # Compute loss between predictions and true labels

    optimizer.zero_grad()        # Clear gradients from the previous step
    loss.backward()              # Backward pass: compute gradients
    optimizer.step()             # Update model weights using the gradients
# Trains for 100 epochs, each time performing forward pass, loss computation, and backpropagation.

### Training the Model

Training in PyTorch follows a manual loop with four key steps per epoch:

1. **Forward pass** — `outputs = model(X)`: Passes the input through the network and gets predictions.
2. **Loss computation** — `loss = criterion(outputs, y)`: Computes how wrong the predictions are compared to the true labels.
3. **Zero gradients** — `optimizer.zero_grad()`: Clears the gradients accumulated from the previous epoch. This is necessary because PyTorch accumulates gradients by default.
4. **Backward pass + update** — `loss.backward()` followed by `optimizer.step()`: Computes the gradient of the loss with respect to every parameter using backpropagation, then updates each weight and bias in the direction that reduces the loss.

With each of the 100 epochs, the optimizer nudges the weights to reduce the `BCELoss`, gradually teaching the network the AND function.

In [ ]:
# 6. Predictions
print("\nPredictions:")
with torch.no_grad():  # Disable gradient computation during inference
    predictions = model(X)
    for i, pred in enumerate(predictions):
        print(f"Input: {X[i].tolist()} => Predicted: {round(float(pred), 4)} => Class: {int(pred >= 0.5)}")
# torch.no_grad(): Disables gradient tracking to save memory during inference.
# A threshold of 0.5 is used to convert the sigmoid output to class labels (0 or 1).


Predictions:
Input: [0.0, 0.0] => Predicted: 0.2082 => Class: 0
Input: [0.0, 1.0] => Predicted: 0.2371 => Class: 0
Input: [1.0, 0.0] => Predicted: 0.3124 => Class: 0
Input: [1.0, 1.0] => Predicted: 0.7344 => Class: 1


### Result

| Input | Expected Output (AND) | Predicted Probability | Predicted Class |
|-------|------------------------|----------------------|------------------|
| `[0.0, 0.0]` | 0 | ~0.2082 | 0 |
| `[0.0, 1.0]` | 0 | ~0.2371 | 0 |
| `[1.0, 0.0]` | 0 | ~0.3124 | 0 |
| `[1.0, 1.0]` | 1 | ~0.7344 | 1 |

* **`torch.no_grad()`**: Wrapping inference in this context manager disables gradient computation, which is unnecessary during prediction and saves both memory and compute time.
* The **Sigmoid output** gives a probability between 0 and 1. A threshold of **0.5** converts this to a binary class label — values ≥ 0.5 are classified as `1`, and values < 0.5 as `0`.
* After 100 epochs of training, the model correctly identifies `[1.0, 1.0]` as the only input pair that produces a `1` — matching the AND gate truth table exactly.
* This demonstrates how even a small feedforward neural network (2 linear layers with ReLU and Sigmoid activations) can learn a simple logical function through gradient-based optimization in PyTorch.